# Task:2 Predicting Future Sales (Regression Model)

## Wallmart Sales Forecasting

In [20]:
# importing necessary libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    root_mean_squared_error,
    mean_absolute_error,
    r2_score
)

## Exploratory Data Analysis (EDA)

In [31]:
# load the dataset
df_train = pd.read_csv('/content/train.csv')
df_stores = pd.read_csv('/content/stores.csv')
df_features = pd.read_csv('/content/features.csv')

In [32]:
df_train.head(4)

,Store,Dept,Date,Weekly_Sales,IsHoliday
0,1,1,2010-02-05,24924.50,False
1,1,1,2010-02-12,46039.49,True
2,1,1,2010-02-19,41595.55,False
3,1,1,2010-02-26,19403.54,False


In [29]:
df_stores.head(4)

,Store,Type,Size
0,1,A,151315
1,2,A,202307
2,3,B,37392
3,4,A,205863


In [30]:
df_features.head(4)

,Store,Date,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,IsHoliday
0,1,2010-02-05,42.31,2.572,NaN,NaN,NaN,NaN,NaN,211.096358,8.106,False
1,1,2010-02-12,38.51,2.548,NaN,NaN,NaN,NaN,NaN,211.242170,8.106,True
2,1,2010-02-19,39.93,2.514,NaN,NaN,NaN,NaN,NaN,211.289143,8.106,False
3,1,2010-02-26,46.63,2.561,NaN,NaN,NaN,NaN,NaN,211.319643,8.106,False


In [36]:
df = df_train.merge(df_stores, on=['Store'])

In [41]:
df = df_train.merge(df_features, on=['Store','Date'])

In [43]:
df.head()

,Store,Dept,Date,Weekly_Sales,IsHoliday_x,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,IsHoliday_y
0,1,1,2010-02-05,24924.50,False,42.31,2.572,NaN,NaN,NaN,NaN,NaN,211.096358,8.106,False
1,1,1,2010-02-12,46039.49,True,38.51,2.548,NaN,NaN,NaN,NaN,NaN,211.242170,8.106,True
2,1,1,2010-02-19,41595.55,False,39.93,2.514,NaN,NaN,NaN,NaN,NaN,211.289143,8.106,False
3,1,1,2010-02-26,19403.54,False,46.63,2.561,NaN,NaN,NaN,NaN,NaN,211.319643,8.106,False
4,1,1,2010-03-05,21827.90,False,46.50,2.625,NaN,NaN,NaN,NaN,NaN,211.350143,8.106,False


In [44]:
df.shape

(421570, 15)

In [42]:
df.isnull().sum()

,0
Store,0
Dept,0
Date,0
Weekly_Sales,0
IsHoliday_x,0
Temperature,0
Fuel_Price,0
MarkDown1,270889
MarkDown2,310322
MarkDown3,284479


In [45]:
# Mrkdown1,2,3,4,5 has more than 50% missing values so better to remove
# and also holiday column appear twice
df.drop(['IsHoliday_x','MarkDown1','MarkDown2','MarkDown3','MarkDown4','MarkDown5'],
        axis=1,inplace=True)

In [46]:
df.head(2)

,Store,Dept,Date,Weekly_Sales,Temperature,Fuel_Price,CPI,Unemployment,IsHoliday_y
0,1,1,2010-02-05,24924.50,42.31,2.572,211.096358,8.106,False
1,1,1,2010-02-12,46039.49,38.51,2.548,211.242170,8.106,True


In [47]:
df.rename(columns={"IsHoliday_y":"IsHoliday"},inplace=True)

In [48]:
# check top few rows
df.head()

,Store,Dept,Date,Weekly_Sales,Temperature,Fuel_Price,CPI,Unemployment,IsHoliday
0,1,1,2010-02-05,24924.50,42.31,2.572,211.096358,8.106,False
1,1,1,2010-02-12,46039.49,38.51,2.548,211.242170,8.106,True
2,1,1,2010-02-19,41595.55,39.93,2.514,211.289143,8.106,False
3,1,1,2010-02-26,19403.54,46.63,2.561,211.319643,8.106,False
4,1,1,2010-03-05,21827.90,46.50,2.625,211.350143,8.106,False


# 🏪 Walmart Weekly Sales Dataset — Column Descriptions

| **Column Name** | **Description** |
|------------------|------------------|
| **Store** | The unique **store ID** — identifies which Walmart store the record belongs to. |
| **Dept** | The **department number** within the store (e.g., Electronics, Grocery, Clothing). |
| **Date** | The **week-ending date** (YYYY-MM-DD format) for that record’s sales data. Each row represents one week of sales. |
| **Weekly_Sales** | The **total sales (in dollars)** for that store and department during that week. |
| **Temperature** | The **average temperature (°F)** in the region of the store during that week — can affect customer shopping behavior. |
| **Fuel_Price** | The **average cost of fuel (per gallon)** in the region — often used as an economic indicator. |
| **CPI** | **Consumer Price Index** — measures inflation; helps track changes in the cost of living over time. |
| **Unemployment** | The **unemployment rate (%)** in the store’s region during that week — reflects local economic conditions. |
| **IsHoliday** | A **Boolean flag (True/False)** indicating whether that week includes a major U.S. holiday (e.g., Thanksgiving, Christmas, Super Bowl). |


In [52]:
# change date column to date time
df['Date'] = pd.to_datetime(df['Date'])

In [55]:
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Week'] = df['Date'].dt.isocalendar().week
df['Day'] = df['Date'].dt.day
df['DayOfWeek'] = df['Date'].dt.dayofweek

In [51]:
df['Date'].dt.isocalender().week

AttributeError: Can only use .dt accessor with datetimelike values

In [56]:
df['Temperature'].unique()

array([42.31, 38.51, 39.93, ..., 75.87, 77.55, 74.09])

In [57]:
# check count of missing values in each columns
df.isnull().sum()

,0
Store,0
Dept,0
Date,0
Weekly_Sales,0
Temperature,0
Fuel_Price,0
CPI,0
Unemployment,0
IsHoliday,0
Year,0


In [58]:
# quick summary of the descriptive statistics
df.describe()

,Store,Dept,Date,Weekly_Sales,Temperature,Fuel_Price,CPI,Unemployment,Year,Month,Week,Day,DayOfWeek
count,421570.000000,421570.000000,421570,421570.000000,421570.000000,421570.000000,421570.000000,421570.000000,421570.000000,421570.000000,421570.0,421570.000000,421570.0
mean,22.200546,44.260317,2011-06-18 08:30:31.963375104,15981.258123,60.090059,3.361027,171.201947,7.960289,2010.968591,6.449510,25.826762,15.673131,4.0
min,1.000000,1.000000,2010-02-05 00:00:00,-4988.940000,-2.060000,2.472000,126.064000,3.879000,2010.000000,1.000000,1.0,1.000000,4.0
25%,11.000000,18.000000,2010-10-08 00:00:00,2079.650000,46.680000,2.933000,132.022667,6.891000,2010.000000,4.000000,14.0,8.000000,4.0
50%,22.000000,37.000000,2011-06-17 00:00:00,7612.030000,62.090000,3.452000,182.318780,7.866000,2011.000000,6.000000,26.0,16.000000,4.0
75%,33.000000,74.000000,2012-02-24 00:00:00,20205.852500,74.280000,3.738000,212.416993,8.572000,2012.000000,9.000000,38.0,23.000000,4.0
max,45.000000,99.000000,2012-10-26 00:00:00,693099.360000,100.140000,4.468000,227.232807,14.313000,2012.000000,12.000000,52.0,31.000000,4.0
std,12.785297,30.492054,NaN,22711.183519,18.447931,0.458515,39.159276,1.863296,0.796876,3.243217,14.151887,8.753549,0.0


In [59]:
# no. of rows and columns in dataframe
df.shape

(421570, 14)

In [60]:
# name of columns
df.columns

Index(['Store', 'Dept', 'Date', 'Weekly_Sales', 'Temperature', 'Fuel_Price',
       'CPI', 'Unemployment', 'IsHoliday', 'Year', 'Month', 'Week', 'Day',
       'DayOfWeek'],
      dtype='object')

In [61]:
# data types of each column
df.dtypes

,0
Store,int64
Dept,int64
Date,datetime64[ns]
Weekly_Sales,float64
Temperature,float64
Fuel_Price,float64
CPI,float64
Unemployment,float64
IsHoliday,bool
Year,int32


In [62]:
# unique values in store column
df['Store'].unique()

array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
       18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34,
       35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45])

In [63]:
df['Dept'].unique()

array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 16, 17, 18,
       19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35,
       36, 37, 38, 40, 41, 42, 44, 45, 46, 47, 48, 49, 51, 52, 54, 55, 56,
       58, 59, 60, 67, 71, 72, 74, 77, 78, 79, 80, 81, 82, 83, 85, 87, 90,
       91, 92, 93, 94, 95, 96, 97, 98, 99, 39, 50, 43, 65])

In [64]:
df.head(10)

,Store,Dept,Date,Weekly_Sales,Temperature,Fuel_Price,CPI,Unemployment,IsHoliday,Year,Month,Week,Day,DayOfWeek
0,1,1,2010-02-05,24924.50,42.31,2.572,211.096358,8.106,False,2010,2,5,5,4
1,1,1,2010-02-12,46039.49,38.51,2.548,211.242170,8.106,True,2010,2,6,12,4
2,1,1,2010-02-19,41595.55,39.93,2.514,211.289143,8.106,False,2010,2,7,19,4
3,1,1,2010-02-26,19403.54,46.63,2.561,211.319643,8.106,False,2010,2,8,26,4
4,1,1,2010-03-05,21827.90,46.50,2.625,211.350143,8.106,False,2010,3,9,5,4
5,1,1,2010-03-12,21043.39,57.79,2.667,211.380643,8.106,False,2010,3,10,12,4
6,1,1,2010-03-19,22136.64,54.58,2.720,211.215635,8.106,False,2010,3,11,19,4
7,1,1,2010-03-26,26229.21,51.45,2.732,211.018042,8.106,False,2010,3,12,26,4
8,1,1,2010-04-02,57258.43,62.27,2.719,210.820450,7.808,False,2010,4,13,2,4
9,1,1,2010-04-09,42960.91,65.86,2.770,210.622857,7.808,False,2010,4,14,9,4


In [68]:
df.corr()

,Store,Dept,Date,Weekly_Sales,Temperature,Fuel_Price,CPI,Unemployment,IsHoliday,Year,Month,Week,Day,DayOfWeek
Store,1.000000,0.024004,0.003362,-0.085195,-0.050097,0.065290,-0.211088,0.208552,-0.000548,0.002997,0.001011,0.001031,-0.000015,NaN
Dept,0.024004,1.000000,0.004054,0.148032,0.004437,0.003572,-0.007477,0.007837,0.000916,0.003738,0.000904,0.000882,-0.000678,NaN
Date,0.003362,0.004054,1.000000,-0.000663,0.147064,0.771913,0.077001,-0.243370,-0.013017,0.941467,0.146422,0.160332,0.041757,NaN
Weekly_Sales,-0.085195,0.148032,-0.000663,1.000000,-0.002312,-0.000120,-0.020921,-0.025864,0.012774,-0.010111,0.028409,0.027673,-0.006187,NaN
Temperature,-0.050097,0.004437,0.147064,-0.002312,1.000000,0.143859,0.182112,0.096730,-0.155949,0.065814,0.235983,0.236276,0.026832,NaN
Fuel_Price,0.065290,0.003572,0.771913,-0.000120,0.143859,1.000000,-0.164210,-0.033853,-0.078281,0.779633,-0.040876,-0.031140,0.028058,NaN
CPI,-0.211088,-0.007477,0.077001,-0.020921,0.182112,-0.164210,1.000000,-0.299953,-0.001944,0.074544,0.005282,0.006342,0.002744,NaN
Unemployment,0.208552,0.007837,-0.243370,-0.025864,0.096730,-0.033853,-0.299953,1.000000,0.010460,-0.237161,-0.012444,-0.015490,-0.003793,NaN
IsHoliday,-0.000548,0.000916,-0.013017,0.012774,-0.155949,-0.078281,-0.001944,0.010460,1.000000,-0.056746,0.123376,0.128184,0.045465,NaN
Year,0.002997,0.003738,0.941467,-0.010111,0.065814,0.779633,0.074544,-0.237161,-0.056746,1.000000,-0.194288,-0.181797,0.005835,NaN


In [66]:
df['DayOfWeek'].dtypes

dtype('int32')

In [67]:
df.dtypes

,0
Store,int64
Dept,int64
Date,datetime64[ns]
Weekly_Sales,float64
Temperature,float64
Fuel_Price,float64
CPI,float64
Unemployment,float64
IsHoliday,bool
Year,int32
